# 03 — Business Analysis (Pandas version of the SQL queries)
Same 9 business questions as `sql/analysis_queries.sql`, done with pandas `groupby` instead of SQL.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/processed/opd_visits_clean.csv')
df.shape

## 1. Average wait time by department

In [ ]:
q1 = df.groupby('Department')['WaitTimeMinutes'].agg(avg_wait_minutes='mean', total_visits='count').round(1)
q1 = q1.sort_values('avg_wait_minutes', ascending=False)
q1

## 2. Wait time by hour of arrival (peak hours)

In [ ]:
q2 = df.groupby('ArrivalHour')['WaitTimeMinutes'].agg(avg_wait_minutes='mean', total_visits='count').round(1)
q2 = q2.sort_index()
q2

## 3. Wait time by day of week

In [ ]:
q3 = df.groupby('DayOfWeek')['WaitTimeMinutes'].agg(avg_wait_minutes='mean', total_visits='count').round(1)
q3 = q3.sort_values('avg_wait_minutes', ascending=False)
q3

## 4. Top 10 longest individual wait times

In [ ]:
q4 = df[['PatientID', 'Department', 'VisitDate', 'TriageCategory', 'WaitTimeMinutes']]
q4 = q4.sort_values('WaitTimeMinutes', ascending=False).head(10)
q4

## 5. Department + hour combo with worst wait times

In [ ]:
q5 = df.groupby(['Department', 'ArrivalHour'])['WaitTimeMinutes'].agg(avg_wait_minutes='mean', total_visits='count').round(1)
q5 = q5[q5['total_visits'] > 5].sort_values('avg_wait_minutes', ascending=False).head(10)
q5

## 6. Wait time by triage urgency category

In [ ]:
q6 = df.groupby('TriageCategory')['WaitTimeMinutes'].agg(avg_wait_minutes='mean', total_visits='count').round(1)
q6 = q6.sort_values('avg_wait_minutes', ascending=False)
q6

## 7. Impact of staffing levels on wait time

In [ ]:
q7 = df.groupby(['ProvidersOnShift', 'NursesOnShift'])['WaitTimeMinutes'].agg(avg_wait_minutes='mean', total_visits='count').round(1)
q7 = q7.sort_values('avg_wait_minutes', ascending=False).head(10)
q7

## 8. Weekend vs. weekday comparison

In [ ]:
q8 = df.groupby('IsWeekend')['WaitTimeMinutes'].agg(avg_wait_minutes='mean', total_visits='count').round(1)
q8

## 9. Overall KPI summary

In [ ]:
kpi = {
    'total_patients': len(df),
    'avg_wait_minutes': round(df['WaitTimeMinutes'].mean(), 1),
    'max_wait_minutes': df['WaitTimeMinutes'].max(),
    'min_wait_minutes': df['WaitTimeMinutes'].min(),
    'avg_total_time_in_hospital': round(df['TotalTimeInHospital'].mean(), 1),
}
kpi

## Bonus: visualize the two most useful charts for your dashboard/README

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14,5))

q1['avg_wait_minutes'].plot(kind='bar', ax=axes[0], title='Avg Wait Time by Department')
axes[0].set_ylabel('Minutes')

q2['avg_wait_minutes'].plot(kind='line', marker='o', ax=axes[1], title='Avg Wait Time by Arrival Hour')
axes[1].set_ylabel('Minutes')
axes[1].set_xlabel('Hour of Day')

plt.tight_layout()
plt.savefig('../reports/screenshots/wait_time_overview.png', dpi=150)
plt.show()